# 🚀 DBT: source() vs ref() --- Complete Detailed Guide

This is one of the **most important DBT concepts** 🔥\
👉 Frequently asked in interviews.

------------------------------------------------------------------------

# 🧠 Core Difference

  --------------------------------------------------------------------------
  Feature      ref()                         source()
  ------------ ----------------------------- -------------------------------
  Used For     DBT models                    Raw/source tables

  Purpose      Build DAG between models      Declare external data sources

  Dependency   Internal (DBT-managed)        External (outside DBT)

  Usage        Transformations               Ingestion layer
  --------------------------------------------------------------------------

------------------------------------------------------------------------

# 🧩 1. What is ref()?

👉 `ref()` is used to reference **another DBT model**

------------------------------------------------------------------------

## 📌 Example

``` sql
SELECT *
FROM {{ ref('stg_orders') }}
```

------------------------------------------------------------------------

## 🧠 What Happens Internally

-   Resolves table name\
-   Adds dependency in DAG\
-   Ensures correct execution order

------------------------------------------------------------------------

## 🔗 Example Flow

stg_orders → int_orders → mart_orders

------------------------------------------------------------------------

# 🧩 2. What is source()?

👉 `source()` is used to reference **raw data (external tables)**

------------------------------------------------------------------------

## 📌 Step 1: Define Source

``` yaml
sources:
  - name: raw
    tables:
      - name: orders
```

------------------------------------------------------------------------

## 📌 Step 2: Use in Model

``` sql
SELECT *
FROM {{ source('raw', 'orders') }}
```

------------------------------------------------------------------------

## 🧠 What Happens Internally

-   Points to external table\
-   Enables source freshness checks\
-   Tracks lineage from raw data

------------------------------------------------------------------------

# 🔥 Key Concept

source() → entry point (raw data)\
ref() → transformation flow (models)

------------------------------------------------------------------------

# 🏗️ Real Production Example

------------------------------------------------------------------------

## Step 1: Source Layer

``` yaml
sources:
  - name: raw
    tables:
      - name: orders
      - name: customers
```

------------------------------------------------------------------------

## Step 2: Staging Model

``` sql
-- stg_orders.sql

SELECT
    id,
    customer_id,
    amount
FROM {{ source('raw', 'orders') }}
```

------------------------------------------------------------------------

## Step 3: Intermediate Model

``` sql
-- int_orders.sql

SELECT
    customer_id,
    SUM(amount) AS total_spent
FROM {{ ref('stg_orders') }}
GROUP BY customer_id
```

------------------------------------------------------------------------

## Step 4: Mart Model

``` sql
-- mart_customer.sql

SELECT
    customer_id,
    total_spent
FROM {{ ref('int_orders') }}
```

------------------------------------------------------------------------

# 🧠 DAG Representation

raw.orders → stg_orders → int_orders → mart_customer

------------------------------------------------------------------------

# ⚙️ When to Use What?

------------------------------------------------------------------------

## ✅ Use source() when:

-   Reading raw tables\
-   Working with ingestion data\
-   Defining external dependencies

------------------------------------------------------------------------

## ✅ Use ref() when:

-   Referencing DBT models\
-   Building transformations\
-   Creating DAG

------------------------------------------------------------------------

# ⚠️ Common Mistake

❌ Wrong:

``` sql
SELECT * FROM raw.orders
```

👉 No lineage, no dependency

------------------------------------------------------------------------

✅ Correct:

``` sql
SELECT * FROM {{ source('raw', 'orders') }}
```

------------------------------------------------------------------------

# 🧠 Advanced Features

------------------------------------------------------------------------

## 📊 Source Freshness

``` yaml
sources:
  - name: raw
    tables:
      - name: orders
        freshness:
          warn_after: {count: 1, period: day}
```

------------------------------------------------------------------------

## 📈 Lineage Tracking

👉 DBT shows: - Raw → staging → marts

------------------------------------------------------------------------

# 🎯 Interview Answer (Short)

👉 `source()`: - Used for raw/external tables

👉 `ref()`: - Used for DBT models\
- Builds DAG

------------------------------------------------------------------------

# ⚡ Final Summary

-   `source()` = raw data entry\
-   `ref()` = transformation flow\
-   Both are essential for lineage and dependency tracking
